# 📦 Notebook 1 — Data Loading & Cleaning

---

## What we did

We collected data from **7 international datasets** covering 192 countries and merged them into a single master file. Think of this as assembling a global spreadsheet where each row is a country and each column tells us something different about that country's climate risk, poverty level, inequality, or social protection coverage.

**The 7 datasets used:**

| Dataset | What it measures | Source |
|--------|-----------------|--------|
| **ND-GAIN** | How vulnerable and ready each country is to climate change | Notre Dame University |
| **WRI** | Physical climate risk — floods, storms, droughts | Bochum University |
| **ILO ILOSTAT** | What % of people are covered by social protection | UN Labour Organisation |
| **World Bank ASPIRE** | Social protection benefit adequacy | World Bank |
| **WDI** | GDP, poverty, life expectancy, health spending | World Bank |
| **WIID** | Income inequality — Gini coefficients, income shares | UN University |
| **UNICEF** | Adaptive social protection mechanisms | UNICEF (unavailable) |

---

## Data challenges we encountered

- Several files were **Excel files disguised as CSVs** — required special handling to open
- The **WID database** originally consisted of 400+ individual country files that crashed the kernel when merged — solved by downloading a pre-aggregated version
- **Country names differed between datasets** — for example ILO uses "United Kingdom of Great Britain and Northern Ireland" while the World Bank uses "United Kingdom" — required a manual name-matching dictionary of 180+ countries
- **UNICEF data was unavailable** as a clean public dataset — replaced with a placeholder and flagged as a data gap throughout the analysis

---

## What we ended up with

> **192 countries · 30 variables · saved to `outputs/master.csv`**

The master dataset includes climate vulnerability scores, social protection coverage rates, GDP, inequality measures, life expectancy, health expenditure, and UN region and income group classifications for every country in the world with available data.

---

## Key data limitation to know

**41% of countries have no social protection coverage data at all.** This is not a random gap — it is one of the findings of the project. The countries missing SP data tend to be among the most vulnerable. We return to this in Notebook 3 as Hypothesis 5.

In [2]:
#Cell 1 — Imports


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [3]:
pip install statsmodels 

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install scikit-learn 

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install scipy 

Note: you may need to restart the kernel to use updated packages.


In [7]:
#install plotly and geopandas
import plotly.express as px
#import geopandas as gpd

In [8]:
#Cell 2 — Country Name Harmonisation
ISO3_MAP = {
    'Afghanistan':'AFG','Albania':'ALB','Algeria':'DZA','Angola':'AGO',
    'Argentina':'ARG','Armenia':'ARM','Australia':'AUS','Austria':'AUT',
    'Azerbaijan':'AZE','Bangladesh':'BGD','Belarus':'BLR','Belgium':'BEL',
    'Benin':'BEN','Bolivia':'BOL','Bolivia (Plurinational State of)':'BOL',
    'Bosnia and Herzegovina':'BIH','Brazil':'BRA','Bulgaria':'BGR',
    'Burkina Faso':'BFA','Burundi':'BDI','Cambodia':'KHM','Cameroon':'CMR',
    'Canada':'CAN','Chad':'TCD','Chile':'CHL','China':'CHN','Colombia':'COL',
    'Congo, Dem. Rep.':'COD','Congo, Rep.':'COG',"Cote d'Ivoire":'CIV',
    'Croatia':'HRV','Cuba':'CUB','Czech Republic':'CZE','Denmark':'DNK',
    'Dominican Republic':'DOM','Ecuador':'ECU','Egypt':'EGY','El Salvador':'SLV',
    'Ethiopia':'ETH','Finland':'FIN','France':'FRA','Ghana':'GHA','Greece':'GRC',
    'Guatemala':'GTM','Guinea':'GIN','Haiti':'HTI','Honduras':'HND',
    'Hungary':'HUN','India':'IND','Indonesia':'IDN','Iran':'IRN',
    'Iran, Islamic Rep.':'IRN','Iraq':'IRQ','Ireland':'IRL','Israel':'ISR',
    'Italy':'ITA','Jamaica':'JAM','Japan':'JPN','Jordan':'JOR',
    'Kazakhstan':'KAZ','Kenya':'KEN','Korea, Rep.':'KOR','Kosovo':'XKX',
    'Kyrgyz Republic':'KGZ','Lao PDR':'LAO','Lebanon':'LBN','Libya':'LBY',
    'Madagascar':'MDG','Malawi':'MWI','Malaysia':'MYS','Mali':'MLI',
    'Mauritania':'MRT','Mexico':'MEX','Moldova':'MDA','Mongolia':'MNG',
    'Morocco':'MAR','Mozambique':'MOZ','Myanmar':'MMR','Namibia':'NAM',
    'Nepal':'NPL','Netherlands':'NLD','New Zealand':'NZL','Nicaragua':'NIC',
    'Niger':'NER','Nigeria':'NGA','North Macedonia':'MKD','Norway':'NOR',
    'Pakistan':'PAK','Palestine':'PSE','Panama':'PAN','Papua New Guinea':'PNG',
    'Paraguay':'PRY','Peru':'PER','Philippines':'PHL','Poland':'POL',
    'Portugal':'PRT','Romania':'ROU','Russia':'RUS','Rwanda':'RWA',
    'Saudi Arabia':'SAU','Senegal':'SEN','Sierra Leone':'SLE',
    'Slovak Republic':'SVK','Somalia':'SOM','South Africa':'ZAF',
    'South Sudan':'SSD','Spain':'ESP','Sri Lanka':'LKA','Sudan':'SDN',
    'Sweden':'SWE','Switzerland':'CHE','Syrian Arab Republic':'SYR',
    'Tajikistan':'TJK','Tanzania':'TZA','Thailand':'THA','Togo':'TGO',
    'Trinidad and Tobago':'TTO','Tunisia':'TUN','Turkey':'TUR','Turkiye':'TUR',
    'Uganda':'UGA','Ukraine':'UKR','United Kingdom':'GBR',
    'United States':'USA','Uruguay':'URY','Uzbekistan':'UZB',
    'Venezuela':'VEN','Venezuela, RB':'VEN','Viet Nam':'VNM',
    'Vietnam':'VNM','Yemen':'YEM','Yemen, Rep.':'YEM','Zambia':'ZMB',
    'Zimbabwe':'ZWE','Micronesia, Fed. Sts.':'FSM','Vanuatu':'VUT',
    'Kiribati':'KIR','Tuvalu':'TUV','Marshall Islands':'MHL',
    'Timor-Leste':'TLS',
}

def to_iso3(name):
    if pd.isna(name):
        return np.nan
    return ISO3_MAP.get(str(name).strip(), np.nan)

print("ISO3 mapping ready")

ISO3 mapping ready


In [9]:
import zipfile
import os

zip_files = [
    'data/raw/ndgain.csv',
    'data/raw/world_risk_index_2023.csv',
    'data/raw/wid_income_shares.csv',
]

for path in zip_files:
    if os.path.exists(path):
        try:
            with zipfile.ZipFile(path, 'r') as z:
                print(f"\n{path} is a zip containing:")
                for name in z.namelist():
                    print(f"  {name}")
                z.extractall('data/raw/')
                print(f"  ✓ Extracted to data/raw/")
        except zipfile.BadZipFile:
            print(f"{path} — not a zip file, skipping")


data/raw/ndgain.csv is a zip containing:
  resources/readiness/readiness.csv
  resources/readiness/social.csv
  resources/readiness/economic.csv
  resources/readiness/governance.csv
  resources/readiness/readiness_delta.csv
  resources/gain/gain_delta.csv
  resources/gain/gain.csv
  resources/trends/readiness.csv
  resources/trends/vulnerability.csv
  resources/trends/gain.csv
  resources/indicators/id_infr_04/input.csv
  resources/indicators/id_infr_04/raw.csv
  resources/indicators/id_infr_04/score.csv
  resources/indicators/id_infr_04/raw0.csv
  resources/indicators/id_soci_04/input.csv
  resources/indicators/id_soci_04/raw.csv
  resources/indicators/id_soci_04/score.csv
  resources/indicators/id_soci_04/raw0.csv
  resources/indicators/id_wate_01/input.csv
  resources/indicators/id_wate_01/raw.csv
  resources/indicators/id_wate_01/score.csv
  resources/indicators/id_wate_01/raw0.csv
  resources/indicators/id_heal_06/input.csv
  resources/indicators/id_heal_06/raw.csv
  resources/in

In [10]:
print("Files now in data/raw/:")
for f in sorted(os.listdir('data/raw')):
    size = os.path.getsize(f'data/raw/{f}')
    print(f"  {f}  ({size:,} bytes)")

Files now in data/raw/:
  ilostat_social_protection.csv  (8,839,566 bytes)
  inform_risk_2024.csv  (2,187,071 bytes)
  ndgain.csv  (4,465,993 bytes)
  resources  (4,096 bytes)
  wdi_1960-2024.csv  (1,668,739 bytes)
  wid_all.csv.csv  (5,264,892 bytes)
  world_bank_aspire.csv  (7,665 bytes)
  world_risk_index_2023.csv  (2,539,923 bytes)
  worldriskindex-datasets  (8,192 bytes)


In [11]:
import pandas as pd
import numpy as np
import os
%pip install openpyxl

def load_file(path, **kwargs):
    for enc in ['utf-8','latin-1','cp1252']:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except:
            continue
    return pd.DataFrame()


"""# ── UNICEF placeholder ────────────────────────────────────────────────────────
unicef = pd.DataFrame(columns=['country','iso3','adaptive_sp_score',
                                'has_trigger','has_registry'])
print("\nUNICEF: placeholder (empty)")

print("\n✓ All datasets loaded. Now print full column names:")
for name, df in [('ndgain',ndgain),('inform',inform),('wri',wri),
                  ('ilo_sp',ilo_sp),('aspire',aspire),('wdi',wdi),('wid_raw',wid_raw)]:
    print(f"\n{name}: {list(df.columns)}")"""

Note: you may need to restart the kernel to use updated packages.


'# ── UNICEF placeholder ────────────────────────────────────────────────────────\nunicef = pd.DataFrame(columns=[\'country\',\'iso3\',\'adaptive_sp_score\',\n                                \'has_trigger\',\'has_registry\'])\nprint("\nUNICEF: placeholder (empty)")\n\nprint("\n✓ All datasets loaded. Now print full column names:")\nfor name, df in [(\'ndgain\',ndgain),(\'inform\',inform),(\'wri\',wri),\n                  (\'ilo_sp\',ilo_sp),(\'aspire\',aspire),(\'wdi\',wdi),(\'wid_raw\',wid_raw)]:\n    print(f"\n{name}: {list(df.columns)}")'

In [ ]:
# Rename wid_raw to wid for consistency
wid = wid_raw.copy()

# Now check all column names
for name, df in [('ndgain', ndgain), ('inform', inform), ('wri', wri),
                 ('ilo_sp', ilo_sp), ('aspire', aspire), ('wdi', wdi),
                 ('unicef', unicef), ('wid', wid)]:
    if not df.empty:
        print(f"\n{'='*40}")
        print(f"{name}: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        print(df.head(2))
    else:
        print(f"\n{name}: EMPTY")


ndgain: (192, 31)
Columns: ['ISO3', 'Name', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023']
  ISO3         Name       1995       1996       1997       1998       1999  \
0  AFG  Afghanistan  34.859009  34.850554  34.927518  35.095338  35.086928   
1  ALB      Albania  44.057068  44.039788  43.973939  43.720561  43.425650   

        2000       2001       2002  ...       2014       2015       2016  \
0  35.082468  35.485125  35.891927  ...  32.918922  33.055386  33.355206   
1  43.559159  43.794627  44.222181  ...  49.729500  50.204170  49.887834   

        2017       2018       2019       2020       2021       2022       2023  
0  32.606196  32.473637  33.027896  32.721670  33.005301  33.087691  33.192894  
1  49.843041  50.329419  50.272865  50.608773  51.442776  51.878407  52.125679  

[2 rows x 31 co

In [13]:
#Cell 4 — Clean and Reshape Each Dataset
# ── ISO3 mapping (self-contained so no dependency on Cell 2) ─────────────────
ISO3_MAP = {
    'Afghanistan':'AFG','Albania':'ALB','Algeria':'DZA','Angola':'AGO',
    'Argentina':'ARG','Armenia':'ARM','Australia':'AUS','Austria':'AUT',
    'Azerbaijan':'AZE','Bangladesh':'BGD','Belarus':'BLR','Belgium':'BEL',
    'Benin':'BEN','Bolivia':'BOL','Bolivia (Plurinational State of)':'BOL',
    'Bosnia and Herzegovina':'BIH','Brazil':'BRA','Bulgaria':'BGR',
    'Burkina Faso':'BFA','Burundi':'BDI','Cambodia':'KHM','Cameroon':'CMR',
    'Canada':'CAN','Chad':'TCD','Chile':'CHL','China':'CHN','Colombia':'COL',
    'Congo, Dem. Rep.':'COD','Congo, Rep.':'COG',"Cote d'Ivoire":'CIV',
    'Croatia':'HRV','Cuba':'CUB','Czech Republic':'CZE','Denmark':'DNK',
    'Dominican Republic':'DOM','Ecuador':'ECU','Egypt':'EGY','El Salvador':'SLV',
    'Ethiopia':'ETH','Finland':'FIN','France':'FRA','Ghana':'GHA','Greece':'GRC',
    'Guatemala':'GTM','Guinea':'GIN','Haiti':'HTI','Honduras':'HND',
    'Hungary':'HUN','India':'IND','Indonesia':'IDN','Iran':'IRN',
    'Iran, Islamic Rep.':'IRN','Iraq':'IRQ','Ireland':'IRL','Israel':'ISR',
    'Italy':'ITA','Jamaica':'JAM','Japan':'JPN','Jordan':'JOR',
    'Kazakhstan':'KAZ','Kenya':'KEN','Korea, Rep.':'KOR','Kosovo':'XKX',
    'Kyrgyz Republic':'KGZ','Lao PDR':'LAO','Lebanon':'LBN','Libya':'LBY',
    'Madagascar':'MDG','Malawi':'MWI','Malaysia':'MYS','Mali':'MLI',
    'Mauritania':'MRT','Mexico':'MEX','Moldova':'MDA','Mongolia':'MNG',
    'Morocco':'MAR','Mozambique':'MOZ','Myanmar':'MMR','Namibia':'NAM',
    'Nepal':'NPL','Netherlands':'NLD','New Zealand':'NZL','Nicaragua':'NIC',
    'Niger':'NER','Nigeria':'NGA','North Macedonia':'MKD','Norway':'NOR',
    'Pakistan':'PAK','Palestine':'PSE','Panama':'PAN','Papua New Guinea':'PNG',
    'Paraguay':'PRY','Peru':'PER','Philippines':'PHL','Poland':'POL',
    'Portugal':'PRT','Romania':'ROU','Russia':'RUS','Rwanda':'RWA',
    'Saudi Arabia':'SAU','Senegal':'SEN','Sierra Leone':'SLE',
    'Slovak Republic':'SVK','Somalia':'SOM','South Africa':'ZAF',
    'South Sudan':'SSD','Spain':'ESP','Sri Lanka':'LKA','Sudan':'SDN',
    'Sweden':'SWE','Switzerland':'CHE','Syrian Arab Republic':'SYR',
    'Tajikistan':'TJK','Tanzania':'TZA','Thailand':'THA','Togo':'TGO',
    'Trinidad and Tobago':'TTO','Tunisia':'TUN','Turkey':'TUR','Turkiye':'TUR',
    'Uganda':'UGA','Ukraine':'UKR','United Kingdom':'GBR',
    'United States':'USA','Uruguay':'URY','Uzbekistan':'UZB',
    'Venezuela':'VEN','Venezuela, RB':'VEN','Viet Nam':'VNM',
    'Vietnam':'VNM','Yemen':'YEM','Yemen, Rep.':'YEM','Zambia':'ZMB',
    'Zimbabwe':'ZWE','Micronesia, Fed. Sts.':'FSM','Vanuatu':'VUT',
    'Kiribati':'KIR','Tuvalu':'TUV','Marshall Islands':'MHL',
    'Timor-Leste':'TLS',
}

def to_iso3(name):
    if pd.isna(name):
        return np.nan
    return ISO3_MAP.get(str(name).strip(), np.nan)



In [14]:
import pandas as pd
import numpy as np
import os
%pip install openpyxl

def load_file(path, **kwargs):
    for enc in ['utf-8','latin-1','cp1252']:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except:
            continue
    return pd.DataFrame()

# ── ND-GAIN: use the main gain score file ────────────────────────────────────
ndgain = load_file('data/raw/resources/gain/gain.csv')
print(f"ND-GAIN: {ndgain.shape} | columns: {list(ndgain.columns[:6])}")

Note: you may need to restart the kernel to use updated packages.
ND-GAIN: (192, 31) | columns: ['ISO3', 'Name', '1995', '1996', '1997', '1998']


In [15]:
# ── 1. ND-GAIN ────────────────────────────────────────────────────────────────
ndgain_clean = ndgain[['ISO3', 'Name', '2023']].copy()
ndgain_clean.columns = ['iso3', 'country', 'gain_score']
ndgain_clean['vulnerability'] = 100 - ndgain_clean['gain_score']
print(f"ND-GAIN clean: {ndgain_clean.shape}")



ND-GAIN clean: (192, 4)


In [16]:
# ── INFORM ───────────────────────────────────────────────────────────────────
inform = pd.read_excel('data/raw/inform_risk_2024.csv', engine='openpyxl')
print(f"INFORM: {inform.shape}")
print(f"Columns: {list(inform.columns[:10])}")
print(inform.head(2))

# ── 2. INFORM: skip ───────────────────────────────────────────────────────────

inform_clean = pd.DataFrame()
print("INFORM: skipped")

INFORM: (72, 1)
Columns: ['Unnamed: 0']
               Unnamed: 0
0                release:
1  31 August 2025 v 0.7.1
INFORM: skipped


In [17]:
# ── WRI: use 2023 file specifically ──────────────────────────────────────────
wri = load_file('data/raw/worldriskindex-datasets/worldriskindex-2023.csv')
print(f"WRI: {wri.shape} | columns: {list(wri.columns[:6])}")

# ── 3. WRI ────────────────────────────────────────────────────────────────────
wri_clean = wri[['ISO3.Code','WRI.Country','W','E','V','S','C','A']].copy()
wri_clean.columns = ['iso3','country_wri','wri_score','exposure',
                     'vulnerability_wri','susceptibility',
                     'lack_coping','lack_adaptation']
print(f"WRI clean: {wri_clean.shape}")

WRI: (193, 248) | columns: ['WRI.Country', 'ISO3.Code', 'Year', 'W', 'E', 'V']
WRI clean: (193, 8)


In [18]:
# ── ILO Social Protection ─────────────────────────────────────────────────────
ilo_sp = load_file('data/raw/ilostat_social_protection.csv')
print(f"ILO SP: {ilo_sp.shape} | columns: {list(ilo_sp.columns)}")

# ── 4. ILO SP ─────────────────────────────────────────────────────────────────
ilo_clean = ilo_sp[
    (ilo_sp['sex.label'] == 'Total') &
    (ilo_sp['indicator.label'].str.contains('1.3.1', na=False)) &
    (ilo_sp['classif1.label'].str.contains('at least one', na=False))
].copy()

ilo_clean = (ilo_clean
             .sort_values('time', ascending=False)
             .groupby('ref_area.label')
             .first()
             .reset_index())

ilo_clean = ilo_clean[['ref_area.label','obs_value','time']].copy()
ilo_clean.columns = ['country_ilo','sp_coverage_rate','sp_year']
ilo_clean['iso3'] = ilo_clean['country_ilo'].apply(to_iso3)
print(f"ILO SP clean: {ilo_clean.shape}")

ILO SP: (36163, 8) | columns: ['ref_area.label', 'source.label', 'indicator.label', 'sex.label', 'classif1.label', 'time', 'obs_value', 'note_classif.label']
ILO SP clean: (300, 4)


In [19]:
# ── ASPIRE ────────────────────────────────────────────────────────────────────
aspire = load_file('data/raw/world_bank_aspire.csv')
print(f"ASPIRE: {aspire.shape} | columns: {list(aspire.columns[:6])}")
# ── 5. ASPIRE: skip ───────────────────────────────────────────────────────────
aspire_clean = pd.DataFrame()
print("ASPIRE: skipped")

ASPIRE: (0, 0) | columns: []
ASPIRE: skipped


In [20]:
# ── WDI ───────────────────────────────────────────────────────────────────────
wdi = load_file('data/raw/wdi_1960-2024.csv')
print(f"WDI: {wdi.shape} | columns: {list(wdi.columns[:6])}")

# ── 6. WDI ────────────────────────────────────────────────────────────────────
wdi_latest = (wdi
              .sort_values('year', ascending=False)
              .groupby('country_code')
              .first()
              .reset_index())

wdi_clean = wdi_latest[[
    'country_code','country','year',
    'gdp_per_capita','poverty_ratio',
    'unemployment_rate','life_expectancy',
    'health_expenditure_pct_gdp'
]].copy()
wdi_clean.columns = [
    'iso3','country_wdi','wdi_year',
    'gdp_per_capita','poverty_ratio',
    'unemployment_rate','life_expectancy','health_exp_gdp'
]
print(f"WDI clean: {wdi_clean.shape}")

WDI: (17425, 13) | columns: ['country', 'country_code', 'year', 'education_expenditure_pct_gdp', 'gdp_current_usd', 'gdp_per_capita']
WDI clean: (261, 8)


In [23]:
# WIID alternative — already clean, one row per country-year
wid_raw = pd.read_csv(f"C:\\Users\\User\\Documents\\Iron Hack\\FINAL PROJECT\\Final\\data\\raw\\wid_all.csv.csv", encoding='latin-1')
print(f"WID: {wid_raw.shape}")
print(f"Columns: {list(wid_raw.columns)}")
print(wid_raw.head(3))

WID: (11826, 60)
Columns: ['Unnamed: 0', 'id', 'country', 'c3', 'c2', 'year', 'gini_reported', 'palma', 'ratio_top20bottom20', 'bottom40', 'q1', 'q2', 'q3', 'q4', 'q5', 'd1', 'd2', 'd3', 'd4', 'd5', 'd6', 'd7', 'd8', 'd9', 'd10', 'bottom5', 'top5', 'resource', 'resource_detailed', 'scale', 'scale_detailed', 'sharing_unit', 'reference_unit', 'areacovr', 'areacovr_detailed', 'popcovr', 'popcovr_detailed', 'region_un', 'region_un_sub', 'region_wb', 'eu', 'oecd', 'incomegroup', 'mean', 'median', 'currency', 'reference_period', 'exchangerate', 'mean_usd', 'median_usd', 'gdp_ppp_pc_usd2011', 'population', 'revision', 'quality', 'quality_score', 'source', 'source_detailed', 'source_comments', 'survey', 'link']
   Unnamed: 0  id      country   c3  c2  year  gini_reported  palma  \
0           0   1  Afghanistan  AFG  AF  2008           29.0    NaN   
1           1   2  Afghanistan  AFG  AF  2012           33.0    NaN   
2           2   3  Afghanistan  AFG  AF  2017           31.0    NaN   

  

In [24]:
#cleaning WID

# ── WID clean: take most recent year per country ──────────────────────────────
wid_clean = (wid_raw
             .sort_values('year', ascending=False)
             .groupby('c3')  # c3 is the ISO3 code column
             .first()
             .reset_index())

wid_clean = wid_clean[[
    'c3', 'country', 'year',
    'gini_reported',   # overall inequality measure
    'bottom40',        # income share of bottom 40%
    'q1',              # bottom quintile share (bottom 20%)
    'q5',              # top quintile share (top 20%)
    'd10',             # top decile share (top 10%)
    'palma',           # palma ratio: top 10% / bottom 40%
    'incomegroup',     # World Bank income group
    'region_un',       # UN region
    'region_wb',       # World Bank region
    'mean_usd',        # mean income in USD
]].copy()

wid_clean.columns = [
    'iso3', 'country_wid', 'wid_year',
    'gini', 'bottom40_share', 'bottom20_share',
    'top20_share', 'top10_share', 'palma_ratio',
    'income_group', 'region_un', 'region_wb',
    'mean_income_usd'
]

print(f"WID clean: {wid_clean.shape}")
print(f"Countries: {wid_clean['iso3'].nunique()}")
print(f"Year range: {wid_clean['wid_year'].min()} - {wid_clean['wid_year'].max()}")
print(wid_clean.head(3))

WID clean: (200, 13)
Countries: 200
Year range: 1977 - 2018
  iso3  country_wid  wid_year   gini  bottom40_share  bottom20_share  \
0  AFG  Afghanistan      2017  31.00           22.00            9.00   
1  AGO       Angola      2009  55.00           10.00            3.00   
2  ALB      Albania      2012  28.96           22.02            8.85   

   top20_share  top10_share  palma_ratio         income_group region_un  \
0        40.00          NaN          NaN           Low income      Asia   
1        59.00        32.31         2.15  Lower middle income    Africa   
2        37.82        22.93         1.04  Upper middle income    Europe   

                 region_wb  mean_income_usd  
0               South Asia              NaN  
1       Sub-Saharan Africa              NaN  
2  Europe and Central Asia              NaN  


In [27]:
# ── CELL 5: Build Master Dataframe ───────────────────────────────────────────

# Start with ND-GAIN as backbone
master = ndgain_clean[['iso3', 'country', 'gain_score', 'vulnerability']].copy()
print(f"Start: {master.shape}")

# Merge WRI
master = master.merge(
    wri_clean[['iso3','wri_score','exposure','susceptibility',
               'lack_coping','lack_adaptation']],
    on='iso3', how='left'
)
print(f"After WRI: {master.shape}")

# Merge ILO SP
master = master.merge(
    ilo_clean[['iso3','sp_coverage_rate','sp_year']],
    on='iso3', how='left'
)
print(f"After ILO: {master.shape}")

# Merge WDI
master = master.merge(
    wdi_clean[['iso3','gdp_per_capita','poverty_ratio',
               'unemployment_rate','life_expectancy','health_exp_gdp']],
    on='iso3', how='left'
)
print(f"After WDI: {master.shape}")

# Merge WID
master = master.merge(
    wid_clean[['iso3','gini','bottom40_share','bottom20_share',
               'top20_share','top10_share','palma_ratio',
               'income_group','region_un','region_wb','mean_income_usd']],
    on='iso3', how='left'
)
print(f"After WID: {master.shape}")

# Add UNICEF placeholders
master['adaptive_sp_score'] = np.nan
master['has_trigger']       = np.nan
master['has_registry']      = np.nan

# Log GDP
master['log_gdp'] = np.log(master['gdp_per_capita'].replace(0, np.nan))

# Summary
print(f"\n✓ Master shape: {master.shape}")
print(f"✓ Countries: {master['iso3'].nunique()}")
print(f"\nMissingness (%):")
print((master.isnull().mean() * 100).round(1).sort_values(ascending=False))

# Preview countries with most complete data
print(f"\nSample — countries with SP coverage data:")
print(master[master['sp_coverage_rate'].notna()]
      [['country','vulnerability','wri_score',
        'sp_coverage_rate','gini','gdp_per_capita']]
      .head(8).to_string(index=False))

# Save
master.to_csv('outputs/master.csv', index=False)
print("\n✓ Saved to outputs/master.csv")

Start: (192, 4)
After WRI: (192, 9)
After ILO: (192, 11)
After WDI: (192, 16)
After WID: (192, 26)

✓ Master shape: (192, 30)
✓ Countries: 192

Missingness (%):
has_registry         100.0
has_trigger          100.0
adaptive_sp_score    100.0
mean_income_usd       82.3
sp_year               41.1
sp_coverage_rate      41.1
poverty_ratio         12.5
palma_ratio            8.9
top10_share            8.9
unemployment_rate      7.8
bottom20_share         6.2
bottom40_share         6.2
top20_share            6.2
gini                   3.6
region_un              2.6
income_group           2.6
region_wb              2.6
vulnerability          2.6
gain_score             2.6
health_exp_gdp         0.5
log_gdp                0.5
gdp_per_capita         0.5
country                0.0
life_expectancy        0.0
lack_adaptation        0.0
lack_coping            0.0
susceptibility         0.0
exposure               0.0
wri_score              0.0
iso3                   0.0
dtype: float64

Sample — coun